# White blood cell classification – Colab quickstart
Runtime → Change runtime type → **T4 GPU** before running.
Run the cells top to bottom. Training results are copied to Google Drive at the end so they survive a disconnect.

In [ ]:
!nvidia-smi
import torch; print("CUDA available:", torch.cuda.is_available())

## 1. Get the code
If the repo is private, use a personal access token in the URL: `https://<TOKEN>@github.com/...`

In [ ]:
REPO = "https://github.com/Yasmin-maker1/Automated-Classification-of-White-Blood-Cell-Subtypes-Using-Convolutional-Neural-Networks.git"
!git clone {REPO} repo
%cd repo
!pip -q install scikit-learn pandas matplotlib

## 2. Get the data (Kaggle)
Create a token at Kaggle → Settings → *Create New Token* and upload the `kaggle.json` file when prompted.

In [ ]:
from google.colab import files
files.upload()   # choose kaggle.json
!mkdir -p ~/.kaggle && mv kaggle.json ~/.kaggle/ && chmod 600 ~/.kaggle/kaggle.json
!pip -q install kaggle
!kaggle datasets download -d paultimothymooney/blood-cells -p data --unzip

## 3. Explore the data and check for leakage

In [ ]:
!python src/eda.py --data-root data
from IPython.display import Image, display
display(Image("outputs/eda/sample_grid.png"))
display(Image("outputs/eda/class_counts.png"))

In [ ]:
!python src/leakage_check.py --data-root data --num-workers 2
display(Image("outputs/leakage/top_pairs.png"))      # LOOK at these
display(Image("outputs/leakage/random_pairs.png"))

## 3b. Group-aware split (fixes the leaky validation set)
Matches every image to the original photo it was made from (`dataset-master/JPEGImages`, rotated and flipped every 5°), then splits train/val **by photo**. Also flags the test images whose photo is in train. About 5–10 min on a T4. The split file is saved to Google Drive, so run this **once** and reuse it.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
DRIVE = "/content/drive/MyDrive/wbc_project"
SPLIT = f"{DRIVE}/group_split.json"
import os
if os.path.exists(SPLIT):
    print("Split file already on Drive - reusing it:", SPLIT)
else:
    !python src/group_split.py --data-root data --out {SPLIT} --fig-dir {DRIVE}/group_split --num-workers 2

In [ ]:
from IPython.display import Image, display
for f in ["match_quality.png", "weakest_matches.png", "random_matches.png", "test_overlap_examples.png"]:
    path = f"{DRIVE}/group_split/{f}"
    if os.path.exists(path):
        print(f); display(Image(path))

## 4. Train
Baseline first, then the main model. Compare **validation** curves; do not use the test folder yet.

In [ ]:
!python src/train.py --data-root data --model smallcnn --epochs 20 --num-workers 2 --amp --run-name smallcnn_baseline

In [ ]:
!python src/train.py --data-root data --model resnet50 --epochs 15 --freeze-epochs 2 --num-workers 2 --amp --run-name resnet50_v1
display(Image("outputs/resnet50_v1/curves.png"))

## 4b. Model comparison: ResNet-50 vs EfficientNet-B0 vs DenseNet-169 (+ frozen backbone, baseline)
Same recipe for every model; only the backbone changes. 3 seeds each, 15 runs, on the group split from 3b, roughly 2.5–3.5 h on a T4.
Results go straight to Google Drive. If Colab disconnects, re-run the mount cell and this cell: finished runs are skipped.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
SPLIT = "/content/drive/MyDrive/wbc_project/group_split.json"   # made in section 3b
EXP = "/content/drive/MyDrive/wbc_project/experiments_groupsplit"
!python src/run_experiments.py --data-root data --out-dir {EXP} --split-file {SPLIT} --amp --num-workers 2 --dry-run   # check the plan first

In [ ]:
!python src/run_experiments.py --data-root data --out-dir {EXP} --split-file {SPLIT} --amp --num-workers 2

In [ ]:
!python src/aggregate_runs.py --out-dir {EXP}
for run in ["resnet50_ft_s42", "efficientnet_b0_ft_s42", "densenet169_ft_s42", "resnet50_frozen_s42"]:
    print(run); display(Image(f"{EXP}/{run}/curves.png"))

## 5. FINAL evaluation (run once, for the chosen model only)

In [ ]:
# !python src/evaluate.py --checkpoint outputs/resnet50_v1/best.pt --num-workers 2
# display(Image("outputs/resnet50_v1/eval/confusion_matrix.png"))

## 6. Save results to Google Drive

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
!mkdir -p /content/drive/MyDrive/wbc_project && cp -r outputs /content/drive/MyDrive/wbc_project/